# 统一入口：下载数据 → 调参 → Clean Expert → Robust Fine-tuning → Ensemble 提交

这个 notebook 是当前项目的统一主入口，用来完成：

1. 下载并整理 robust fine-tuning 需要的西式数字数据集；
2. 小样本 HPO / 调参，输出 `outputs_submission/hpo/best_params.json`；
3. 训练或加载 Clean Expert（MNIST / QMNIST / EMNIST digits / USPS）；
4. 从 clean checkpoint 初始化并 fine-tune Robust Expert；
5. 在 validation board 上比较 clean / robust / ensemble；
6. 对考试图片执行 clean + robust 概率融合推理。

默认不会覆盖 `outputs_submission/checkpoints/best_model_state*.pt` 这些历史高分 checkpoint。Robust checkpoint 使用新文件名 `robust_expert_best.pt`。

In [1]:
from pathlib import Path
import importlib
import json
import sys

import pandas as pd
import torch

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path(r"E:\ALL\学习\AI导论作业-识别手写数字")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.download_finetune_data as download_finetune_data
import src.ensemble_predict as ensemble_predict
import src.evaluate as evaluate_module
import src.hpo as hpo_module
import src.robust_train as robust_train
import src.validation_board as validation_board

for module in [download_finetune_data, ensemble_predict, evaluate_module, hpo_module, robust_train, validation_board]:
    importlib.reload(module)

from src.config import ExperimentConfig, ensure_project_paths
from src.data import create_dataloaders
from src.download_finetune_data import prepare_all as prepare_finetune_datasets
from src.engine import fit
from src.ensemble_predict import predict_image_folder, search_ensemble_weight
from src.evaluate import (
    collect_predictions,
    evaluate_external_holdouts,
    evaluate_mnist_c_zip,
    load_model_from_checkpoint,
    save_evaluation_bundle,
)
from src.hpo import run_hpo
from src.model import build_model, count_model_parameters
from src.predict import PredictionImageDataset, predict_with_tta, write_predictions_csv
from src.robust_train import run_robust_finetune
from src.train import set_seed
from src.validation_board import evaluate_validation_board

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PROJECT_ROOT, DEVICE

(WindowsPath('e:/ALL/学习/AI导论作业-识别手写数字'), 'cuda')

## 统一运行控制面板

只需要改下面一个 cell 的开关，然后运行后面的“统一执行入口”。

推荐顺序：

1. 第一次运行：`DOWNLOAD_FINETUNE_DATASETS=True`，先把 HASYv2 / Chars74K / UCI Pen-Based 整理到 `data/` 下；
2. 需要调参时：`RUN_HPO=True`，先用小样本和少量 epoch 搜索，再把结果写入 `outputs_submission/hpo/`；
3. 保留现有 clean 高分模型：`TRAIN_CLEAN_MODEL=False`，直接使用 `outputs_submission/checkpoints/best_model_state_09974.pt`；
4. 微调 robust：`TRAIN_ROBUST_MODEL=True`；
5. 测试/验证：`RUN_VALIDATION_BOARD=True`、`RUN_HOLDOUTS=True`；
6. 搜索 ensemble 权重并预测：`RUN_ENSEMBLE_INFERENCE=True`，需要先设置 `EXAM_IMAGE_DIR`。

In [2]:
DOWNLOAD_FINETUNE_DATASETS = False
FORCE_REDOWNLOAD_DATASETS = False

RUN_HPO = False
HPO_TRIALS = 12
HPO_EPOCHS = 5
HPO_MAX_SAMPLES = 12000

TRAIN_CLEAN_MODEL = False
TRAIN_ROBUST_MODEL = True
RUN_VALIDATION_BOARD = True
RUN_HOLDOUTS = True
RUN_MNIST_C = False
RUN_ENSEMBLE_INFERENCE = False
SEARCH_ENSEMBLE_WEIGHT = True

USE_TTA = True
TTA_N = 8
DEFAULT_ENSEMBLE_CLEAN_WEIGHT = 0.60
ENSEMBLE_WEIGHT_GRID = (0.75, 0.70, 0.65, 0.60, 0.55, 0.50)

OUTPUT_DIR = PROJECT_ROOT / "outputs_submission"
EXAM_IMAGE_DIR = PROJECT_ROOT / "exam_data" / "test"

CHECKPOINT_CANDIDATES = [
    OUTPUT_DIR / "checkpoints" / "best_model_stat-09987e.pt",
    OUTPUT_DIR / "checkpoints" / "checkpoint_clean_best.pth",
    OUTPUT_DIR / "checkpoints" / "best_model_state_09974.pt",
    OUTPUT_DIR / "checkpoints" / "best_model_state.pt",
]
CLEAN_CHECKPOINT = next((path for path in CHECKPOINT_CANDIDATES if path.exists()), CHECKPOINT_CANDIDATES[-1])
ROBUST_CHECKPOINT = OUTPUT_DIR / "checkpoints" / "robust_expert_best.pt"

clean_config = ExperimentConfig(
    project_root=PROJECT_ROOT,
    output_dir=OUTPUT_DIR,
    dataset_name="multisource",
    model_name="medium_cnn",
    use_mnist=True,
    use_emnist_digits=True,
    use_usps=True,
    use_qmnist=True,
    emnist_max_samples=50000,
    qmnist_max_samples=60000,
    validation_source="mixed",
    batch_size=512,
    external_validation_batch_size=512,
    epochs=60,
    seed=42,
    num_workers=8,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
    dataloader_timeout=60,
    dropout=0.21672530847241062,
    optimizer_type="AdamW",
    scheduler_type="CosineAnnealingLR",
    learning_rate=0.0008398721379146775,
    weight_decay=6.602542933207749e-06,
    label_smoothing=0.03,
    rotation_degrees=7.536974266650085,
    translate_ratio=0.05567431908414762,
    scale_min=0.9213414099692713,
    scale_max=1.1010803603468335,
    shear_degrees=4.9755817645420075,
    use_random_affine=True,
    use_gaussian_blur=False,
    use_amp=True,
    allow_tf32=True,
    use_early_stopping=True,
    early_stopping_patience=7,
    early_stopping_min_delta=1e-4,
    external_holdout_names=("mnist_test", "emnist_digits_test", "qmnist_test10k"),
    use_tta=USE_TTA,
    tta_n=TTA_N,
)

robust_config = ExperimentConfig(
    **{**clean_config.to_dict(),
       "training_mode": "robust_finetune",
       "clean_checkpoint_path": str(CLEAN_CHECKPOINT),
       "checkpoint_name": "robust_expert_best.pt",
       "batch_size": 128,
       "learning_rate": 5e-5,
       "fine_tune_lr": 5e-5,
       "fine_tune_epochs": 8,
       "freeze_backbone_first": False,
       "freeze_epochs": 2,
       "optimizer_type": "AdamW",
       "scheduler_type": "CosineAnnealingLR",
       "weight_decay": 1e-5,
       "label_smoothing": 0.03,
       "robust_aug_strength": "medium",
       "use_robust_sampler": True,
       "mnist_family_weight": 0.65,
       "use_local_digits": False,
       "local_digits_dir": str(PROJECT_ROOT / "data" / "local_digits"),
       "local_digits_holdout_dir": str(PROJECT_ROOT / "data" / "local_digits_holdout"),
       "local_digits_weight": 0.00,
       "use_hasyv2": True,
       "hasyv2_dir": str(PROJECT_ROOT / "data" / "hasyv2_digits"),
       "hasyv2_weight": 0.12,
       "use_chars74k": True,
       "chars74k_dir": str(PROJECT_ROOT / "data" / "chars74k_digits"),
       "chars74k_weight": 0.05,
       "use_penbased_rendered": True,
       "penbased_dir": str(PROJECT_ROOT / "data" / "penbased_rendered"),
       "penbased_weight": 0.08,
       "use_optical_digits": False,
       "optical_weight": 0.00,
       "ensemble_weight_clean": DEFAULT_ENSEMBLE_CLEAN_WEIGHT,
       "ensemble_weight_grid": ENSEMBLE_WEIGHT_GRID,
       "use_tta": USE_TTA,
       "tta_n": TTA_N,
    }
)

paths = ensure_project_paths(clean_config)
set_seed(clean_config.seed)

{
    "project_root": str(PROJECT_ROOT),
    "output_dir": str(OUTPUT_DIR),
    "base_clean_checkpoint": str(CLEAN_CHECKPOINT),
    "robust_checkpoint": str(ROBUST_CHECKPOINT),
    "fine_tune_lr": robust_config.fine_tune_lr,
    "fine_tune_epochs": robust_config.fine_tune_epochs,
    "robust_aug_strength": robust_config.robust_aug_strength,
    "ensemble_weight_grid": robust_config.ensemble_weight_grid,
    "exam_image_dir": str(EXAM_IMAGE_DIR),
    "device": DEVICE,
}

{'project_root': 'e:\\ALL\\学习\\AI导论作业-识别手写数字',
 'output_dir': 'e:\\ALL\\学习\\AI导论作业-识别手写数字\\outputs_submission',
 'base_clean_checkpoint': 'e:\\ALL\\学习\\AI导论作业-识别手写数字\\outputs_submission\\checkpoints\\best_model_stat-09987e.pt',
 'robust_checkpoint': 'e:\\ALL\\学习\\AI导论作业-识别手写数字\\outputs_submission\\checkpoints\\robust_expert_best.pt',
 'fine_tune_lr': 5e-05,
 'fine_tune_epochs': 8,
 'robust_aug_strength': 'medium',
 'ensemble_weight_grid': (0.75, 0.7, 0.65, 0.6, 0.55, 0.5),
 'exam_image_dir': 'e:\\ALL\\学习\\AI导论作业-识别手写数字\\exam_data\\test',
 'device': 'cuda'}

## 统一执行入口

运行下面一个 cell 即可按控制面板完成下载、HPO 调参、clean/base training、robust fine-tuning、validation board、holdout 测试、ensemble weight search 和推理。

In [3]:
workflow_results = {
    "dataset_manifest": None,
    "hpo": None,
    "clean_checkpoint": str(CLEAN_CHECKPOINT),
    "robust_checkpoint": str(ROBUST_CHECKPOINT),
    "validation_boards": {},
    "holdouts": None,
    "mnist_c": None,
    "ensemble_weight_search": None,
    "submission_csv": None,
}

if DOWNLOAD_FINETUNE_DATASETS:
    workflow_results["dataset_manifest"] = prepare_finetune_datasets(
        PROJECT_ROOT,
        force=FORCE_REDOWNLOAD_DATASETS,
    )

if RUN_HPO:
    workflow_results["hpo"] = run_hpo(
        clean_config,
        n_trials=HPO_TRIALS,
        trial_epochs=HPO_EPOCHS,
        trial_max_samples=HPO_MAX_SAMPLES,
        device=DEVICE,
    )

if TRAIN_CLEAN_MODEL:
    clean_train_loader, clean_val_loader = create_dataloaders(clean_config)
    clean_model = build_model(clean_config).to(DEVICE)
    clean_history = fit(clean_model, clean_train_loader, clean_val_loader, config=clean_config, paths=paths, device=DEVICE)
    CLEAN_CHECKPOINT = paths.checkpoints_dir / clean_config.checkpoint_name
    workflow_results["clean_checkpoint"] = str(CLEAN_CHECKPOINT)
else:
    if not CLEAN_CHECKPOINT.exists():
        raise FileNotFoundError(f"未找到 clean checkpoint: {CLEAN_CHECKPOINT}")

if TRAIN_ROBUST_MODEL:
    robust_config.clean_checkpoint_path = CLEAN_CHECKPOINT
    run_robust_finetune(robust_config)
    ROBUST_CHECKPOINT = OUTPUT_DIR / "checkpoints" / "robust_expert_best.pt"
    workflow_results["robust_checkpoint"] = str(ROBUST_CHECKPOINT)

clean_eval_model, _ = load_model_from_checkpoint(CLEAN_CHECKPOINT, clean_config, DEVICE)

robust_eval_model = None
if ROBUST_CHECKPOINT.exists():
    robust_eval_model, _ = load_model_from_checkpoint(ROBUST_CHECKPOINT, robust_config, DEVICE)

if RUN_VALIDATION_BOARD:
    workflow_results["validation_boards"]["clean"] = evaluate_validation_board(
        clean_eval_model,
        clean_config,
        OUTPUT_DIR / "logs",
        DEVICE,
        prefix="clean",
    )
    if robust_eval_model is not None:
        workflow_results["validation_boards"]["robust"] = evaluate_validation_board(
            robust_eval_model,
            robust_config,
            OUTPUT_DIR / "logs",
            DEVICE,
            prefix="robust",
        )

if RUN_HOLDOUTS:
    workflow_results["holdouts"] = evaluate_external_holdouts(
        clean_eval_model,
        config=clean_config,
        output_dir=OUTPUT_DIR / "evaluation" / "holdouts_clean",
        device=DEVICE,
    )

if RUN_MNIST_C:
    workflow_results["mnist_c"] = evaluate_mnist_c_zip(
        clean_eval_model,
        config=clean_config,
        output_dir=OUTPUT_DIR / "evaluation" / "mnist_c_clean",
        device=DEVICE,
    )

chosen_weight = DEFAULT_ENSEMBLE_CLEAN_WEIGHT
if robust_eval_model is not None and SEARCH_ENSEMBLE_WEIGHT:
    best_weight_row, weight_rows = search_ensemble_weight(
        clean_eval_model,
        robust_eval_model,
        robust_config,
        DEVICE,
        OUTPUT_DIR / "logs",
    )
    workflow_results["ensemble_weight_search"] = {"best": best_weight_row, "rows": weight_rows}
    if best_weight_row is not None:
        chosen_weight = float(best_weight_row["clean_weight"])

if RUN_ENSEMBLE_INFERENCE:
    if robust_eval_model is None:
        raise FileNotFoundError(f"未找到 robust checkpoint，无法 ensemble: {ROBUST_CHECKPOINT}")
    if not EXAM_IMAGE_DIR.exists():
        raise FileNotFoundError(f"考试图片目录不存在: {EXAM_IMAGE_DIR}")
    prediction_dataset = PredictionImageDataset(
        EXAM_IMAGE_DIR,
        image_size=robust_config.image_size,
        auto_invert=robust_config.auto_invert,
    )
    prediction_loader = torch.utils.data.DataLoader(
        prediction_dataset,
        batch_size=robust_config.batch_size,
        shuffle=False,
    )
    clean_rows, robust_rows, ensemble_rows = predict_image_folder(
        clean_eval_model,
        robust_eval_model,
        prediction_loader,
        robust_config,
        DEVICE,
        clean_weight=chosen_weight,
    )
    write_predictions_csv(clean_rows, OUTPUT_DIR / "predictions" / "clean_predictions.csv")
    write_predictions_csv(robust_rows, OUTPUT_DIR / "predictions" / "robust_predictions.csv")
    write_predictions_csv(ensemble_rows, OUTPUT_DIR / "predictions" / "ensemble_predictions.csv")
    write_predictions_csv(ensemble_rows, OUTPUT_DIR / "submission.csv")
    workflow_results["submission_csv"] = str(OUTPUT_DIR / "submission.csv")

workflow_results

Start training: model=medium_cnn, epochs=8, optimizer=AdamW, scheduler=CosineAnnealingLR, device=cuda
Epoch 001/008 | train_loss=0.5179 train_acc=0.8913 | val_loss=0.2471 val_acc=0.9825 | best_val_acc=0.9825@1 | lr=4.8097e-05 | best | 262.5s
Epoch 002/008 | train_loss=0.4439 train_acc=0.9170 | val_loss=0.2387 val_acc=0.9861 | best_val_acc=0.9861@2 | lr=4.26777e-05 | best | 78.2s
Epoch 003/008 | train_loss=0.4189 train_acc=0.9257 | val_loss=0.2370 val_acc=0.9878 | best_val_acc=0.9878@3 | lr=3.45671e-05 | best | 72.3s
Epoch 004/008 | train_loss=0.4051 train_acc=0.9307 | val_loss=0.2364 val_acc=0.9874 | best_val_acc=0.9878@3 | lr=2.5e-05 | no_improve=1 | 68.4s
Epoch 005/008 | train_loss=0.3970 train_acc=0.9334 | val_loss=0.2359 val_acc=0.9882 | best_val_acc=0.9882@5 | lr=1.54329e-05 | best | 65.3s
Epoch 006/008 | train_loss=0.3891 train_acc=0.9362 | val_loss=0.2338 val_acc=0.9890 | best_val_acc=0.9890@6 | lr=7.32233e-06 | best | 66.3s
Epoch 007/008 | train_loss=0.3889 train_acc=0.9365 | v

e:\ALL\学习\AI导论作业-识别手写数字\src\robust_data.py:253: UserWarning: local_digits 数据目录不存在，已跳过: e:\ALL\学习\AI导论作业-识别手写数字\data\local_digits_holdout
  return _maybe_folder_dataset(
e:\ALL\学习\AI导论作业-识别手写数字\src\robust_data.py:253: UserWarning: local_digits 数据目录不存在，已跳过: e:\ALL\学习\AI导论作业-识别手写数字\data\local_digits_holdout
  return _maybe_folder_dataset(


{'dataset_manifest': None,
 'hpo': None,
 'clean_checkpoint': 'e:\\ALL\\学习\\AI导论作业-识别手写数字\\outputs_submission\\checkpoints\\best_model_stat-09987e.pt',
 'robust_checkpoint': 'e:\\ALL\\学习\\AI导论作业-识别手写数字\\outputs_submission\\checkpoints\\robust_expert_best.pt',
 'validation_boards': {'clean': {'splits': {'val_clean': {'accuracy': 0.998928309549326,
     'num_samples': 35458},
    'val_corrupt_lite': {'accuracy': 0.9952055953522477, 'num_samples': 35458},
    'val_external': {'accuracy': 0.9973, 'num_samples': 60000}},
   'score': {'composite_score': 0.9979628250324327,
    'weights': {'val_clean': 0.6,
     'val_external': 0.25,
     'val_corrupt_lite': 0.15},
    'has_local': False}},
  'robust': {'splits': {'val_clean': {'accuracy': 0.9962490834226408,
     'num_samples': 35458},
    'val_corrupt_lite': {'accuracy': 0.9923571549438772, 'num_samples': 35458},
    'val_external': {'accuracy': 0.980009730713092, 'num_samples': 94546}},
   'score': {'composite_score': 0.9916054559734391,
 

## 统一结果查看

下面的 cell 汇总下载结果、validation board、holdout、ensemble 权重搜索和提交文件路径。详细文件在 `outputs_submission/logs/`、`outputs_submission/evaluation/`、`outputs_submission/predictions/`。

In [4]:
summary_rows = []

manifest = workflow_results.get("dataset_manifest") or {}
for item in manifest.get("results", []):
    summary_rows.append({
        "section": "dataset",
        "name": item["name"],
        "metric": "num_images",
        "value": item["num_images"],
        "path": item["output_dir"],
    })
for item in manifest.get("errors", []):
    summary_rows.append({
        "section": "dataset_error",
        "name": item["name"],
        "metric": "error",
        "value": item["error"],
        "path": "",
    })

hpo_result = workflow_results.get("hpo")
if hpo_result and hpo_result.get("best"):
    best = hpo_result["best"]
    summary_rows.append({
        "section": "hpo",
        "name": "best_trial",
        "metric": "best_val_accuracy",
        "value": best["best_val_accuracy"],
        "path": hpo_result["hpo_dir"],
    })

for board_name, board in workflow_results.get("validation_boards", {}).items():
    score = board.get("score", {})
    summary_rows.append({
        "section": "validation_board",
        "name": board_name,
        "metric": "composite_score",
        "value": score.get("composite_score"),
        "path": str(OUTPUT_DIR / "logs"),
    })
    for split_name, split in board.get("splits", {}).items():
        summary_rows.append({
            "section": f"validation_{board_name}",
            "name": split_name,
            "metric": "accuracy",
            "value": split.get("accuracy"),
            "path": str(OUTPUT_DIR / "logs"),
        })

for item in workflow_results.get("holdouts") or []:
    summary_rows.append({
        "section": "holdout_clean",
        "name": item["name"],
        "metric": "accuracy",
        "value": item["accuracy"],
        "path": str(OUTPUT_DIR / "evaluation" / "holdouts_clean"),
    })

weight_search = workflow_results.get("ensemble_weight_search")
if weight_search and weight_search.get("best"):
    best = weight_search["best"]
    summary_rows.append({
        "section": "ensemble",
        "name": "best_weight",
        "metric": "clean_weight",
        "value": best["clean_weight"],
        "path": str(OUTPUT_DIR / "logs" / "ensemble_weight_search.csv"),
    })

summary_rows.append({
    "section": "submission",
    "name": "submission_csv",
    "metric": "path",
    "value": workflow_results.get("submission_csv"),
    "path": str(OUTPUT_DIR / "submission.csv"),
})

pd.DataFrame(summary_rows)

,section,name,metric,value,path
0,validation_board,clean,composite_score,0.997963,e:\ALL\学习\AI导论作业-识别手写数字\outputs_submission\logs
1,validation_clean,val_clean,accuracy,0.998928,e:\ALL\学习\AI导论作业-识别手写数字\outputs_submission\logs
2,validation_clean,val_corrupt_lite,accuracy,0.995206,e:\ALL\学习\AI导论作业-识别手写数字\outputs_submission\logs
3,validation_clean,val_external,accuracy,0.997300,e:\ALL\学习\AI导论作业-识别手写数字\outputs_submission\logs
4,validation_board,robust,composite_score,0.991605,e:\ALL\学习\AI导论作业-识别手写数字\outputs_submission\logs
5,validation_robust,val_clean,accuracy,0.996249,e:\ALL\学习\AI导论作业-识别手写数字\outputs_submission\logs
6,validation_robust,val_corrupt_lite,accuracy,0.992357,e:\ALL\学习\AI导论作业-识别手写数字\outputs_submission\logs
7,validation_robust,val_external,accuracy,0.980010,e:\ALL\学习\AI导论作业-识别手写数字\outputs_submission\logs
8,holdout_clean,mnist_test,accuracy,0.997300,e:\ALL\学习\AI导论作业-识别手写数字\outputs_submission\eva...
9,holdout_clean,emnist_digits_test,accuracy,0.997300,e:\ALL\学习\AI导论作业-识别手写数字\outputs_submission\eva...


## 数据目录约定

下载整理后，robust fine-tuning 使用这些目录：

- `data/hasyv2_digits/{0..9}/...`
- `data/chars74k_digits/{0..9}/...`
- `data/penbased_rendered/{0..9}/...`
- 可选本地手写：`data/local_digits/{0..9}/...`
- 可选本地 holdout：`data/local_digits_holdout/{0..9}/...`

原始压缩包和解压内容保存在 `data/raw_external/`，整理结果记录在 `data/finetune_datasets_manifest.json`。